In [ ]:
import os
import subprocess
import sys

# NLGCL Kaggle Runner v3
# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Setup Dependencies
print("Configuring environment...")

# Force install compatible Numpy and Pandas FIRST to resolve binary incompatibility
# We pin pandas<2.2.3 because pandas 3.x+ often requires or prefers numpy 2.0+, causing ABI mismatches
# (Fixes: ValueError: numpy.dtype size changed, Expected 96 from C header, got 88)
print("Fixing Numpy/Pandas versions...")
!pip install "numpy<2.0.0" "pandas<2.2.3" --force-reinstall

# FORCE REINSTALL TORCH to a stable version compatible with PyG wheels
# Kaggle's default torch 2.9.0+ is too new and causes slow source compilation for scatter/sparse
print("Downgrading PyTorch to stable 2.4.0...")
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Now we can safely install wheels for PyG
print("Installing PyG wheels...")
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt==0.2.5
torch_geometric
# python>=3.9.18
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing other requirements...")
!pip install -r requirements.txt

# Safe to import now
import torch
import pandas as pd

# 3. Dataset Preprocessing
# Convert CSV to RecBole .inter format with type annotations
inter_path = 'dataset/QB-video/QB-video.inter'
csv_path = 'dataset/QB-video.csv'
source_path = None

if os.path.exists(csv_path):
    source_path = csv_path
elif os.path.exists(inter_path):
    source_path = inter_path

if source_path:
    print(f"Processing dataset from {source_path}...")
    df = pd.read_csv(source_path)
    
    # Check if header needs fixing (missing type annotations)
    if ':' not in str(df.columns[0]):
        print("Detected raw CSV header. Converting to RecBole format (user_id:token, item_id:token)...
,

In [ ]:
import os
import subprocess
import sys

# NLGCL Kaggle Runner v3
# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Setup Dependencies
print("Configuring environment...")

# Force install compatible Numpy and Pandas FIRST to resolve binary incompatibility
# We pin pandas<2.2.3 because pandas 3.x+ often requires or prefers numpy 2.0+, causing ABI mismatches
# (Fixes: ValueError: numpy.dtype size changed, Expected 96 from C header, got 88)
print("Fixing Numpy/Pandas versions...")
!pip install "numpy<2.0.0" "pandas<2.2.3" --force-reinstall

# FORCE REINSTALL TORCH to a stable version compatible with PyG wheels
# Kaggle's default torch 2.9.0+ is too new and causes slow source compilation for scatter/sparse
print("Downgrading PyTorch to stable 2.4.0...")
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Now we can safely install wheels for PyG
print("Installing PyG wheels...")
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt==0.2.5
torch_geometric
# python>=3.9.18
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing other requirements...")
!pip install -r requirements.txt

# Safe to import now
import torch
import pandas as pd

# 3. Dataset Preprocessing
# Convert CSV to RecBole .inter format with type annotations
inter_path = 'dataset/QB-video/QB-video.inter'
csv_path = 'dataset/QB-video.csv'
source_path = None

if os.path.exists(csv_path):
    source_path = csv_path
elif os.path.exists(inter_path):
    source_path = inter_path

if source_path:
    print(f"Processing dataset from {source_path}...")
    df = pd.read_csv(source_path)
    
    # Check if header needs fixing (missing type annotations)
    if ':' not in str(df.columns[0]):
        print("Detected raw CSV header. Converting to RecBole format (user_id:token, item_id:token)...")
        if 'user_id' in df.columns and 'item_id' in df.columns:
            # Keep only required columns and rename with types
            df = df[['user_id', 'item_id']]
            df.columns = ['user_id:token', 'item_id:token']
            
            # Ensure dataset folder exists
            os.makedirs(os.path.dirname(inter_path), exist_ok=True)
            
            # Save using comma separator
            df.to_csv(inter_path, index=False, sep=',')
            print(f"Successfully saved {inter_path}")
    else:
        print("Dataset header already contains type annotations.")
else:
    print("Warning: No dataset file found (checked QB-video.csv and QB-video.inter)")

# 4. Configuration Adjustment
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()
    # Ensure field_separator is set to comma
    if 'field_separator' not in content:
        # Inject field_separator
        new_content = content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\n\nfield_separator: ","')
        with open(config_path, 'w') as f:
            f.write(new_content)
        print("Updated properties/QB-video.yaml with field_separator.")

# 5. Run Training
!python main.py --dataset QB-video